# Data Loading

In [0]:
master = spark.read.csv(
    "/Workspace/Users/princebadgoti05@gmail.com/CEI-Assignment/Week_7/data/customer_master.csv",
    header=True,
    inferSchema=True
)

master.show(20)

+----------+-------------+---------+-----------+------+--------+--------+--------------------+
|CustomerID| CustomerName|     City|   Category| Price|Quantity|Discount|               Email|
+----------+-------------+---------+-----------+------+--------+--------+--------------------+
|      1001|Customer_1001|   Mumbai|Electronics|2353.0|       4|       5|customer1001@exam...|
|      1002|Customer_1002|   Jaipur|Electronics|4567.0|       2|      15|customer1002@exam...|
|      1003|Customer_1003|    Delhi|Electronics| 867.0|       4|       5|customer1003@exam...|
|      1004|Customer_1004|  Lucknow|     Sports| 317.0|       9|       5|customer1004@exam...|
|      1005|Customer_1005|  Lucknow|    Grocery|1905.0|       8|      10|customer1005@exam...|
|      1006|Customer_1006|     NULL|  Furniture|3562.0|       6|      10|customer1006@exam...|
|      1007|Customer_1007|   Jaipur|  Furniture|2857.0|       2|       0|customer1007@exam...|
|      1008|Customer_1008|  Chennai|Electronics|30

In [0]:
master.count()

103

# Handling Duplicates and Null Values

In [0]:

master.groupBy("CustomerID").count().filter("count > 1").show()

+----------+-----+
|CustomerID|count|
+----------+-----+
|      1021|    2|
|      1004|    2|
|      1051|    2|
+----------+-----+



In [0]:

total_count = master.count()
distinct_count = master.distinct().count()
duplicate_count = total_count - distinct_count

print(f"Total rows: {total_count}")
print(f"Distinct rows: {distinct_count}")
print(f"Duplicate rows: {duplicate_count}")

Total rows: 103
Distinct rows: 100
Duplicate rows: 3


In [0]:
from pyspark.sql.functions import col,sum

master.select([sum(col(c).isNull().cast("int")).alias(c) for c in master.columns]).show()

+----------+------------+----+--------+-----+--------+--------+-----+
|CustomerID|CustomerName|City|Category|Price|Quantity|Discount|Email|
+----------+------------+----+--------+-----+--------+--------+-----+
|         0|           0|   4|       0|    2|       0|       0|    0|
+----------+------------+----+--------+-----+--------+--------+-----+



In [0]:
from pyspark.sql.functions import concat, lit, coalesce

avg_price = master.selectExpr("avg(Price) as avg_price").first()["avg_price"]

#  Email nulls with: "customer" + CustomerID + "@examplr.com"
master = master.withColumn(
    "Email",
    coalesce(col("Email"), concat(lit("customer"), col("CustomerID"), lit("@example.com")))
)

#  other nulls
master = master.fillna({"Category": "Unknown", "Price": avg_price})


In [0]:
master.show(20)

+----------+-------------+---------+-----------+-----------------+--------+--------+--------------------+
|CustomerID| CustomerName|     City|   Category|            Price|Quantity|Discount|               Email|
+----------+-------------+---------+-----------+-----------------+--------+--------+--------------------+
|      1001|Customer_1001|   Mumbai|Electronics|           2353.0|       4|       5|customer1001@exam...|
|      1002|Customer_1002|   Jaipur|Electronics|           4567.0|       2|      15|customer1002@exam...|
|      1003|Customer_1003|    Delhi|Electronics|            867.0|       4|       5|customer1003@exam...|
|      1004|Customer_1004|  Lucknow|     Sports|            317.0|       9|       5|customer1004@exam...|
|      1005|Customer_1005|  Lucknow|    Grocery|           1905.0|       8|      10|customer1005@exam...|
|      1006|Customer_1006|     NULL|  Furniture|           3562.0|       6|      10|customer1006@exam...|
|      1007|Customer_1007|   Jaipur|  Furnitur

In [0]:
master.select([sum(col(c).isNull().cast("int")).alias(c) for c in master.columns]).show()

+----------+------------+----+--------+-----+--------+--------+-----+
|CustomerID|CustomerName|City|Category|Price|Quantity|Discount|Email|
+----------+------------+----+--------+-----+--------+--------+-----+
|         0|           0|   4|       0|    0|       0|       0|    0|
+----------+------------+----+--------+-----+--------+--------+-----+



In [0]:

master = master.dropDuplicates()

print(f"Duplicates removed. New count: {master.count()}")

Duplicates removed. New count: 100


In [0]:
# Get total rows vs distinct rows
total_count = master.count()
distinct_count = master.distinct().count()
duplicate_count = total_count - distinct_count

print(f"Total rows: {total_count}")
print(f"Distinct rows: {distinct_count}")
print(f"Duplicate rows: {duplicate_count}")

Total rows: 100
Distinct rows: 100
Duplicate rows: 0


In [0]:
master.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = false)
 |-- Price: double (nullable = false)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Email: string (nullable = true)



## Filter operation applied

In [0]:
master.filter(master.City == "Delhi").show()



+----------+-------------+-----+-----------+------+--------+--------+--------------------+
|CustomerID| CustomerName| City|   Category| Price|Quantity|Discount|               Email|
+----------+-------------+-----+-----------+------+--------+--------+--------------------+
|      1039|Customer_1039|Delhi|Electronics| 582.0|       4|       0|customer1039@exam...|
|      1049|Customer_1049|Delhi|     Sports| 220.0|       2|       5|customer1049@exam...|
|      1009|Customer_1009|Delhi|    Grocery|4492.0|       2|      15|customer1009@exam...|
|      1023|Customer_1023|Delhi|Electronics|1352.0|       3|      15|customer1023@exam...|
|      1082|Customer_1082|Delhi|  Furniture|3364.0|       6|      10|customer1082@exam...|
|      1003|Customer_1003|Delhi|Electronics| 867.0|       4|       5|customer1003@exam...|
|      1038|Customer_1038|Delhi|     Sports|4637.0|       4|       5|customer1038@exam...|
+----------+-------------+-----+-----------+------+--------+--------+--------------------+

In [0]:
# Select Columns
master.select("CustomerID", "CustomerName", "Price").show()

+----------+-------------+-----------------+
|CustomerID| CustomerName|            Price|
+----------+-------------+-----------------+
|      1007|Customer_1007|           2857.0|
|      1011|Customer_1011|            669.0|
|      1018|Customer_1018|           4746.0|
|      1021|Customer_1021|           4880.0|
|      1086|Customer_1086|           2931.0|
|      1090|Customer_1090|           4254.0|
|      1020|Customer_1020|           4698.0|
|      1051|Customer_1051|           1448.0|
|      1058|Customer_1058|           4766.0|
|      1062|Customer_1062|           3854.0|
|      1042|Customer_1042|           3972.0|
|      1048|Customer_1048|           3730.0|
|      1057|Customer_1057|           4974.0|
|      1065|Customer_1065|           3689.0|
|      1097|Customer_1097|           2642.0|
|      1013|Customer_1013|2638.267326732673|
|      1033|Customer_1033|            666.0|
|      1039|Customer_1039|            582.0|
|      1049|Customer_1049|            220.0|
|      107

# New Columns Added (TotalAmount)

In [0]:
master = master.withColumn(
    "total_amount",
    col("Price") * col("Quantity")
)

In [0]:
master.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = false)
 |-- Price: double (nullable = false)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Email: string (nullable = true)
 |-- total_amount: double (nullable = true)



In [0]:
master = master.withColumnRenamed("total_amount", "TotalAmount")
master.printSchema()
# Sort by TotalAmount
master.sort(col("TotalAmount").desc()).show()

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = false)
 |-- Price: double (nullable = false)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Email: string (nullable = true)
 |-- TotalAmount: double (nullable = true)

+----------+-------------+---------+-----------+------+--------+--------+--------------------+-----------+
|CustomerID| CustomerName|     City|   Category| Price|Quantity|Discount|               Email|TotalAmount|
+----------+-------------+---------+-----------+------+--------+--------+--------------------+-----------+
|      1094|Customer_1094|     Pune|    Grocery|4851.0|      10|      10|customer1094@exam...|    48510.0|
|      1054|Customer_1054|     Pune|Electronics|4844.0|       9|       0|customer1054@exam...|    43596.0|
|      1020|Customer_1020|   Jaipur|  Furniture|4698.0|       9|      10|customer1020@exam...| 

In [0]:
master.describe().show()

+-------+------------------+-------------+---------+--------+------------------+-----------------+------------------+--------------------+------------------+
|summary|        CustomerID| CustomerName|     City|Category|             Price|         Quantity|          Discount|               Email|       TotalAmount|
+-------+------------------+-------------+---------+--------+------------------+-----------------+------------------+--------------------+------------------+
|  count|               100|          100|       96|     100|               100|              100|               100|                 100|               100|
|   mean|            1050.5|         NULL|     NULL|    NULL|2650.9653465346537|             5.34|               7.8|                NULL|13945.110693069306|
| stddev|29.011491975882016|         NULL|     NULL|    NULL| 1400.286262659329|2.832780694767328|5.1404515778521285|                NULL| 11059.50444607269|
|    min|              1001|Customer_1001|Bengaluru|

# Delta Table Created

In [0]:

master.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.customer_master")

# Read the Delta table into a DataFrame and display it
delta_df = spark.read.table("workspace.default.customer_master")
display(delta_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email,TotalAmount
1007,Customer_1007,Jaipur,Furniture,2857.0,2,0,customer1007@example.com,5714.0
1011,Customer_1011,Indore,Furniture,669.0,1,5,customer1011@example.com,669.0
1018,Customer_1018,null,Furniture,4746.0,6,5,customer1018@example.com,28476.0
1021,Customer_1021,Indore,Grocery,4880.0,7,10,customer1021@example.com,34160.0
1086,Customer_1086,Indore,Grocery,2931.0,6,15,customer1086@example.com,17586.0
1090,Customer_1090,Mumbai,Clothing,4254.0,5,15,customer1090@example.com,21270.0
1020,Customer_1020,Jaipur,Furniture,4698.0,9,10,customer1020@example.com,42282.0
1051,Customer_1051,Chennai,Electronics,1448.0,7,0,customer1051@example.com,10136.0
1058,Customer_1058,Chennai,Electronics,4766.0,4,0,customer1058@example.com,19064.0
1062,Customer_1062,Mumbai,Electronics,3854.0,10,0,customer1062@example.com,38540.0


# Incremental Dataset Loaded

In [0]:
incremental = spark.read.csv(
    "/Workspace/Users/princebadgoti05@gmail.com/CEI-Assignment/Week_7/data/customer_incremental.csv",
    inferSchema=True,
    header=True
)

display(incremental)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email
1021,Customer_1021,Mumbai,Grocery,5265.0,1,10,customer1021@example.com
1022,Customer_1022,Pune,Furniture,4722.0,7,0,customer1022@example.com
1023,Customer_1023,Kolkata,Electronics,1492.0,9,15,null
1024,Customer_1024,Pune,Electronics,3327.0,9,15,customer1024@example.com
1025,Customer_1025,Kolkata,Clothing,5116.0,2,0,customer1025@example.com
1026,Customer_1026,Lucknow,Clothing,3004.0,8,10,customer1026@example.com
1027,Customer_1027,Hyderabad,Furniture,4152.0,10,10,customer1027@example.com
1028,Customer_1028,Indore,Furniture,4694.0,8,10,customer1028@example.com
1029,Customer_1029,Chennai,Sports,2196.0,9,10,customer1029@example.com
1030,Customer_1030,Kolkata,Sports,4919.0,9,10,customer1030@example.com


## Null Values Checked

In [0]:
incremental.select([sum(col(c).isNull().cast("int")).alias(c) for c in incremental.columns]).show()

+----------+------------+----+--------+-----+--------+--------+-----+
|CustomerID|CustomerName|City|Category|Price|Quantity|Discount|Email|
+----------+------------+----+--------+-----+--------+--------+-----+
|         0|           0|   0|       1|    1|       0|       0|    1|
+----------+------------+----+--------+-----+--------+--------+-----+



### Duplicate values handled

In [0]:
incremental = incremental.dropDuplicates()

# Null Values Handled

In [0]:



from pyspark.sql.functions import concat, lit, coalesce

avg_price = incremental.selectExpr("avg(Price) as avg_price").first()["avg_price"]

#  Email nulls with: "customer" + CustomerID + "@example.com"
incremental = incremental.withColumn(
    "Email",
    coalesce(col("Email"), concat(lit("customer"), col("CustomerID"), lit("@example.com")))
)

#  other nulls
incremental = incremental.fillna({"Category": "Unknown", "Price": avg_price})


In [0]:
incremental.select([sum(col(c).isNull().cast("int")).alias(c) for c in incremental.columns]).show()

+----------+------------+----+--------+-----+--------+--------+-----+
|CustomerID|CustomerName|City|Category|Price|Quantity|Discount|Email|
+----------+------------+----+--------+-----+--------+--------+-----+
|         0|           0|   0|       0|    0|       0|       0|    0|
+----------+------------+----+--------+-----+--------+--------+-----+



# New Columns Added

In [0]:
incremental = incremental.withColumn("TotalAmount", col("Quantity") * col("Price"))
incremental.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = false)
 |-- Price: double (nullable = false)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Email: string (nullable = true)
 |-- TotalAmount: double (nullable = true)



In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(
    spark,
    "workspace.default.customer_master"
   
)

In [0]:
total_count = incremental.count()
distinct_count = incremental.distinct().count()
duplicate_count = total_count-distinct_count
print(f"Total count: {total_count}")
print(f"Distinct count: {distinct_count}")
print(f"Duplicate count: {duplicate_count}")

Total count: 70
Distinct count: 70
Duplicate count: 0


# Merge Operation Applied

In [0]:
from pyspark.sql.functions import col




deltaTable = DeltaTable.forName(spark, "workspace.default.customer_master")

deltaTable.alias("target") \
    .merge(
        incremental.alias("source"),
        "target.CustomerID = source.CustomerID"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()



DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(deltaTable.toDF())

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email,TotalAmount
1007,Customer_1007,Jaipur,Furniture,2857.0,2,0,customer1007@example.com,5714.0
1011,Customer_1011,Indore,Furniture,669.0,1,5,customer1011@example.com,669.0
1018,Customer_1018,null,Furniture,4746.0,6,5,customer1018@example.com,28476.0
1086,Customer_1086,Indore,Grocery,2931.0,6,15,customer1086@example.com,17586.0
1090,Customer_1090,Mumbai,Clothing,4254.0,5,15,customer1090@example.com,21270.0
1020,Customer_1020,Jaipur,Furniture,4698.0,9,10,customer1020@example.com,42282.0
1062,Customer_1062,Mumbai,Electronics,3854.0,10,0,customer1062@example.com,38540.0
1065,Customer_1065,Bengaluru,Furniture,3689.0,9,10,customer1065@example.com,33201.0
1097,Customer_1097,Mumbai,Furniture,2642.0,4,5,customer1097@example.com,10568.0
1013,Customer_1013,Bengaluru,Grocery,2638.267326732673,3,10,customer1013@example.com,7914.801980198019


In [0]:
final_df = spark.read.table("workspace.default.customer_master")

display(final_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email,TotalAmount
1007,Customer_1007,Jaipur,Furniture,2857.0,2,0,customer1007@example.com,5714.0
1011,Customer_1011,Indore,Furniture,669.0,1,5,customer1011@example.com,669.0
1018,Customer_1018,null,Furniture,4746.0,6,5,customer1018@example.com,28476.0
1086,Customer_1086,Indore,Grocery,2931.0,6,15,customer1086@example.com,17586.0
1090,Customer_1090,Mumbai,Clothing,4254.0,5,15,customer1090@example.com,21270.0
1020,Customer_1020,Jaipur,Furniture,4698.0,9,10,customer1020@example.com,42282.0
1062,Customer_1062,Mumbai,Electronics,3854.0,10,0,customer1062@example.com,38540.0
1065,Customer_1065,Bengaluru,Furniture,3689.0,9,10,customer1065@example.com,33201.0
1097,Customer_1097,Mumbai,Furniture,2642.0,4,5,customer1097@example.com,10568.0
1013,Customer_1013,Bengaluru,Grocery,2638.267326732673,3,10,customer1013@example.com,7914.801980198019


In [0]:
from pyspark.sql.functions import col, sum

final_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in final_df.columns
]).show()

+----------+------------+----+--------+-----+--------+--------+-----+-----------+
|CustomerID|CustomerName|City|Category|Price|Quantity|Discount|Email|TotalAmount|
+----------+------------+----+--------+-----+--------+--------+-----+-----------+
|         0|           0|   3|       0|    0|       0|       0|    0|          0|
+----------+------------+----+--------+-----+--------+--------+-----+-----------+



In [0]:
final_df.count()

130

## Final Output

In [0]:
final_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.final_output")

In [0]:


final_df.write.mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/assignment_output/final_output")

print("✓ CSV file successfully written to: /Volumes/workspace/default/assignment_output/final_output/")


✓ CSV file successfully written to: /Volumes/workspace/default/assignment_output/final_output/
